### Filtrado y limpieza: BENIGN y ataques lentos

Prepara el conjunto de datos del proyecto: tráfico benigno y ataques lentos (Slowloris y Slowhttptest) de la captura del miércoles 5/7/2017 de CIC-IDS2017 (`GeneratedLabelledFlows`). Se descartan DoS Hulk, DoS GoldenEye y Heartbleed porque están fuera del alcance del protocolo.

**Etapas**

1. Carga de los datos crudos
2. Filtrado por clase
3. Exclusión de respuestas del servidor a conexiones de ataque
4. Verificación de hipótesis sobre los datos crudos
5. Limpieza básica (incluye la corrección de `Timestamp` y la eliminación de columnas sin valor informativo)
6. Codificación de la etiqueta binaria
7. Exclusión de ataques con etiqueta incoherente en el tiempo
8. Guardado
9. Resumen

Toda la lógica está en `src/preprocessing/clean.py`; este cuaderno solo la llama y reporta resultados. Las cifras del resumen final se imprimen desde el código, no se transcriben a mano.

In [ ]:
import sys
sys.path.append("../..")  # raíz del repo, desde notebooks/Cleaning/

from pathlib import Path

import pandas as pd

from src.preprocessing.clean import (
    load_raw_csvs,
    filter_low_and_slow,
    excluir_respuestas_de_ataque,
    basic_cleaning,
    encode_labels,
    excluir_ataques_fuera_de_ventana,
)

**1. Carga de datos crudos**

`load_raw_csvs` busca de forma recursiva en `data/raw/` el CSV del miércoles que tiene la columna `Timestamp`, necesaria para ordenar los flujos cronológicamente. Deben estar `Flow ID`, `Source IP`, `Destination IP` y `Timestamp`; si faltan, se cargó una versión del dataset que no los trae (`MachineLearningCSV`).

In [ ]:
df_raw = load_raw_csvs("../../data/raw")
print(f"Shape: {df_raw.shape}")

columnas_esperadas = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
faltantes = [c for c in columnas_esperadas if c not in df_raw.columns]
assert not faltantes, (
    f"Faltan columnas esperadas {faltantes} -- parece que se cargo el CSV "
    "equivocado (sin Timestamp/Flow ID/IPs). Revisar data/raw/."
)
print("OK: Flow ID, Source IP, Destination IP y Timestamp presentes.")

Cargado: ..\..\data\raw\TrafficLabelling\Wednesday-workingHours.pcap_ISCX.csv
Shape: (692703, 85)
OK: Flow ID, Source IP, Destination IP y Timestamp presentes.


**2. Filtrado: BENIGN y ataques lentos**

El protocolo se limita a ataques lentos. Se descartan DoS Hulk, DoS GoldenEye y Heartbleed, que son volumétricos o de otra naturaleza, y se conservan el tráfico BENIGN y las clases Slowloris y Slowhttptest.

In [ ]:
df_filtrado = filter_low_and_slow(df_raw)

Etiquetas encontradas en el CSV: ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed']
Filas antes del filtro: 692703 | después (solo BENIGN + Low&Slow): 451326


**3. Exclusión de respuestas del servidor a conexiones de ataque**

En esta captura los atacantes externos aparecen con la IP de la interfaz NAT del firewall (172.16.0.1) y el servidor web víctima es 192.168.10.50. Los 252,661 flujos NAT → víctima están etiquetados como ataque, mientras que los 6,789 flujos BENIGN de ese par van en sentido víctima → NAT. De esos, 6,787 se emparejan con un flujo de ataque por el puerto del cliente (mediana de la diferencia de inicio: 0 s) y todos ocurren solo mientras hay ataques (Slowloris, Slowhttptest, Hulk y GoldenEye). Es decir, son el lado servidor de conexiones de ataque que quedó etiquetado como BENIGN: ruido de etiqueta.

Se excluyen en lugar de reetiquetarse, con el mismo criterio que los ataques fuera de ventana (sección 7). Se hace en este punto porque hacen falta las IPs, que la limpieza básica elimina.

In [ ]:
df_sin_respuestas = excluir_respuestas_de_ataque(df_filtrado)

Respuestas de ataque etiquetadas BENIGN excluidas: 6789 (víctima → NAT: 6789, NAT → víctima: 0; 1.54% de 440031 BENIGN)


**4. Verificación de hipótesis sobre los datos crudos**

Antes de limpiar se comprueban en `df_filtrado` tres propiedades que determinan las decisiones de la sección siguiente:

- **Formato de `Timestamp`.** Si las cadenas no tienen AM/PM y las horas son 1–5 y 8–12 (ninguna de 13 en adelante), la hora está en reloj de 12 horas sin indicador. La captura empieza a las 8:42, así que las horas 1–5 son de la tarde. Ordenar con ese formato pondría la tarde antes que la mañana.
- **Columna duplicada.** `SYN Flag Count` debe coincidir con `Fwd PSH Flags` en el 100 % de las filas; si es así, aportan lo mismo y se conserva una sola.
- **Atajos del entorno.** Se mide la proporción de flujos al puerto 80 y la distribución de protocolos por clase. Si una columna separa las clases por la configuración del laboratorio y no por el comportamiento del tráfico, el modelo puede usarla sin detectar de verdad un ataque lento.

In [ ]:
# Formato del Timestamp: ¿trae AM/PM y qué horas aparecen sin corregir?
ts_crudo = df_filtrado["Timestamp"].astype(str).str.strip()
print("Cadenas con AM/PM:", int(ts_crudo.str.contains(r"\b[AP]M\b", case=False).sum()))
horas_sin_corregir = pd.to_datetime(ts_crudo, dayfirst=True, errors="coerce").dt.hour
print("Horas tal como vienen en el CSV:")
print(horas_sin_corregir.value_counts().sort_index().to_string())

iguales = (df_filtrado["SYN Flag Count"] == df_filtrado["Fwd PSH Flags"]).mean()
print(f"\nSYN Flag Count == Fwd PSH Flags en {iguales:.4%} de las filas")

clase = df_filtrado["Label"].where(df_filtrado["Label"] == "BENIGN", "ATAQUE")
print("\nProporción de flujos al puerto 80, por clase:")
print((df_filtrado["Destination Port"] == 80).groupby(clase).mean().round(4).to_string())
print("\nProtocolo por clase:")
print(pd.crosstab(df_filtrado["Protocol"], clase, normalize="columns").round(4))

Cadenas con AM/PM: 0
Horas tal como vienen en el CSV:
Timestamp
1     48902
2     63868
3     66131
4     44878
5      3547
8     14107
9     54304
10    80545
11    51051
12    23993

SYN Flag Count == Fwd PSH Flags en 100.0000% de las filas

Proporción de flujos al puerto 80, por clase:
Label
ATAQUE    1.0000
BENIGN    0.1108

Protocolo por clase:
Label     ATAQUE  BENIGN
Protocol                
0            0.0  0.0007
6            1.0  0.5381
17           0.0  0.4612


**5. Limpieza básica**

`basic_cleaning` hace lo siguiente:

- Convierte `Timestamp` a fecha y hora (formato D/M/YYYY) y corrige el reloj de 12 horas: a las horas menores que 8 se les suman 12, porque son de la tarde.
- Reemplaza infinitos por valores faltantes y elimina las filas con nulos y los duplicados exactos. Los duplicados se buscan antes de quitar IPs y puertos para eliminar solo copias exactas: dos flujos que difieren únicamente en esas columnas son eventos distintos.
- Elimina las columnas identificadoras (`Flow ID`, IPs, `Source Port`) para evitar fuga de información.
- Elimina los atajos del laboratorio (`Destination Port`, `Protocol`). El 100 % de los ataques va al puerto 80 y todos son TCP, mientras que cerca de la mitad del tráfico benigno es UDP; conservar esas columnas permitiría separar las clases sin modelar el comportamiento lento. Para el experimento de ablación "con atajos" se usa `basic_cleaning(df_sin_respuestas, eliminar_atajos=False)` y se guarda el resultado con otro nombre.
- Elimina las columnas redundantes (entre ellas `Fwd PSH Flags`, idéntica a `SYN Flag Count`) y las constantes, que no aportan información.
- Elimina las filas con valores negativos en duración, tiempos entre paquetes o tasas, que no tienen sentido físico.

La celda termina con tres comprobaciones: `Timestamp` presente y sin valores inválidos, todas las horas dentro del horario de captura (8–17) y flujos de la tarde, lo que confirma que se aplicó la corrección del reloj.

In [ ]:
# Limpieza y comprobación de que el reloj de 12 horas quedó corregido
df_limpio = basic_cleaning(df_sin_respuestas)
print(f"\nShape tras limpieza: {df_limpio.shape}")

assert "Timestamp" in df_limpio.columns, "Timestamp se perdió en basic_cleaning()"
assert df_limpio["Timestamp"].notna().all(), "Hay timestamps no parseables"
horas = df_limpio["Timestamp"].dt.hour
assert horas.between(8, 17).all(), f"Horas fuera de 8-17: {sorted(horas.unique())}"
assert horas.max() >= 13, "No hay flujos de la tarde: la corrección de 12 h no se aplicó"
print("OK: todas las horas están entre 8 y 17 y hay tráfico de la tarde.")

Reloj de 12 h sin AM/PM detectado: 227325 filas con hora < 8 se desplazan +12 h (tarde).
Rango de Timestamp (corregido): 2017-07-05 08:42:00 -> 2017-07-05 17:10:00
Valores nulos/infinitos detectados: 696
Filas duplicadas exactas detectadas: 16
Columnas identificadoras eliminadas: ['Flow ID', 'Source IP', 'Destination IP', 'Source Port']
Columnas atajo de laboratorio eliminadas: ['Destination Port', 'Protocol']
Columnas redundantes eliminadas: ['Subflow Fwd Packets', 'Fwd Header Length.1', 'Avg Fwd Segment Size', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Avg Bwd Segment Size', 'Subflow Bwd Bytes', 'Fwd PSH Flags']
Filas con valores negativos anómalos eliminadas: 554 (0.1247%)
Columnas constantes eliminadas: ['Fwd URG Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Bwd PSH Flags', 'CWE Flag Count']

Shape tras limpieza: (443620, 61)
OK: todas las horas están entre 8 y 17 y h

**6. Etiqueta binaria**

Se crea `label_binaria`: 0 para tráfico BENIGN y 1 para Slowloris o Slowhttptest. La clasificación del proyecto es binaria, pero se conserva `Label` para poder reportar el desempeño por tipo de ataque.

In [ ]:
df_codificado = encode_labels(df_limpio)

**7. Ataques fuera de su ventana documentada**

La descripción del dataset da la ventana horaria de cada ataque: Slowloris de 9:47 a 10:10 y Slowhttptest de 10:14 a 10:35. Se aplica una tolerancia de ±2 minutos porque `Timestamp` tiene granularidad de un minuto.

Slowhttptest queda completo dentro de su ventana. En Slowloris, seis flujos caen fuera: cinco entre las 14:24 y las 14:25, varias horas después de que terminó el ataque, y uno a las 09:01, antes de que empezara.

Una conexión larga iniciada antes del ataque podría explicar una fila fuera de ventana. Pero `Flow Duration` está en microsegundos y estos seis flujos son cortos: cuatro duran cerca de 5,000,000 µs (5 s) y dos duran 229 y 214 µs. Como CICFlowMeter registra el instante de inicio del flujo, ninguno puede extenderse hasta la ventana del ataque.

Se consideran ruido de etiqueta y se excluyen. No se reetiquetan como BENIGN porque su clase real se desconoce. La celda imprime las filas excluidas para tener trazabilidad y se detiene con un error si la exclusión supera el 1 % de los ataques, lo que apuntaría a un problema en `Timestamp` y no a ruido de etiqueta.

In [ ]:
# Rango horario de cada ataque antes y después de excluir las filas fuera de ventana
print("Rango por ataque ANTES de excluir:")
print(df_codificado[df_codificado["label_binaria"] == 1]
      .groupby("Label")["Timestamp"].agg(["min", "max", "count"]))
print()

df_final = excluir_ataques_fuera_de_ventana(df_codificado)

print("\nRango por ataque DESPUÉS de excluir:")
print(df_final[df_final["label_binaria"] == 1]
      .groupby("Label")["Timestamp"].agg(["min", "max", "count"]))

Rango por ataque ANTES de excluir:
                                 min                 max  count
Label                                                          
DoS Slowhttptest 2017-07-05 10:15:00 2017-07-05 10:37:00   5499
DoS slowloris    2017-07-05 09:01:00 2017-07-05 14:25:00   5796

DoS slowloris: 6 de 5796 filas fuera de [2017-07-05 09:45:00 .. 2017-07-05 10:12:00]
                Timestamp  Flow Duration
6558  2017-07-05 14:24:00        5169956
26700 2017-07-05 14:24:00            229
26701 2017-07-05 14:24:00        5001928
26702 2017-07-05 14:24:00        5125872
26703 2017-07-05 14:25:00            214
26704 2017-07-05 09:01:00        5001683
DoS Slowhttptest: 0 de 5499 filas fuera de [2017-07-05 10:12:00 .. 2017-07-05 10:37:00]
Filas excluidas por estar fuera de ventana: 6

Rango por ataque DESPUÉS de excluir:
                                 min                 max  count
Label                                                          
DoS Slowhttptest 2017-07-05 10:15:00

**8. Guardado**

El resultado se guarda en `data/processed/` y es la entrada de los cuadernos del SVM. El archivo se regenera ejecutando este cuaderno sobre la copia de `data/raw/`, cuyo contenido no se modifica.

In [ ]:
out_path = Path("../../data/processed/wednesday_low_and_slow_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(out_path, index=False)
print(f"Guardado: {out_path.resolve()}")

Guardado: C:\Users\juans\OneDrive\Documents\Github\low-and-slow-ids\data\processed\wednesday_low_and_slow_clean.csv


**9. Resumen**

La celda siguiente imprime las cifras del procesamiento: filas tras cada etapa, número de columnas, rango de `Timestamp` y distribución de clases. Toda cifra que se cite en el informe debe tomarse de esta salida.

Para las fases siguientes:

- **Desbalance de clases.** Los ataques son cerca del 2.5 % de los flujos (relación aproximada de 38:1). Debe tratarse con ponderación de clases, submuestreo o SMOTE aplicados solo al entrenamiento, y evaluarse con recall, precisión, F1 y ROC-AUC en lugar de exactitud.
- **Duplicados.** Los duplicados exactos son pocos porque se buscan antes de quitar IPs y puertos, que son casi únicos por flujo. Los vectores de características repetidos, que son los que ve el SVM, se cuantifican después de la partición con `reportar_duplicados_de_features()`.
- **Resolución temporal.** `Timestamp` tiene granularidad de un minuto. Dentro de cada minuto el orden de los flujos es el del archivo CSV, que no siempre coincide con el orden de inicio. El parámetro `desempate` de `train_test_split_by_flow()` permite medir cuánto influye ese orden en los resultados.

In [ ]:
features = [c for c in df_final.columns if c not in ("Timestamp", "Label", "label_binaria")]
conteo = df_final["label_binaria"].value_counts()

print(f"Filas crudas:                   {len(df_raw):,}")
print(f"Tras filtro BENIGN + Low&Slow:  {len(df_filtrado):,}")
print(f"Tras excluir respuestas de ataque: {len(df_sin_respuestas):,}")
print(f"Tras limpieza:                  {len(df_limpio):,}")
print(f"Tras excluir fuera de ventana:  {len(df_final):,}")
print(f"\nColumnas del CSV final: {df_final.shape[1]} "
      f"({len(features)} features + Timestamp + Label + label_binaria)")
print(f"Rango de Timestamp: {df_final['Timestamp'].min()} -> {df_final['Timestamp'].max()}")
print(f"\nBENIGN: {conteo[0]:,} | ataque: {conteo[1]:,} | ratio ≈ {conteo[0]/conteo[1]:.1f} : 1")
print(df_final.loc[df_final["label_binaria"] == 1, "Label"].value_counts().to_string())

Filas crudas:                   692,703
Tras filtro BENIGN + Low&Slow:  451,326
Tras excluir respuestas de ataque: 444,537
Tras limpieza:                  443,620
Tras excluir fuera de ventana:  443,614

Columnas del CSV final: 62 (59 features + Timestamp + Label + label_binaria)
Rango de Timestamp: 2017-07-05 08:42:00 -> 2017-07-05 17:10:00

BENIGN: 432,325 | ataque: 11,289 | ratio ≈ 38.3 : 1
Label
DoS slowloris       5790
DoS Slowhttptest    5499


In [ ]:
from src.preprocessing.clean import cargar_dataset_limpio

df_check = cargar_dataset_limpio(out_path)
assert df_check["Timestamp"].equals(df_final["Timestamp"].reset_index(drop=True)), "Timestamp cambió al recargar"
assert len(df_check) == len(df_final)
print("OK: el CSV se recarga con el mismo Timestamp y el mismo número de filas.")

OK: el CSV se recarga con el mismo Timestamp y el mismo número de filas.
